# 6. Beta and Alpha Diversity Analyses

This notebook performs comprehensive diversity analyses on the NHANES oral microbiome data, including:

- **Alpha Diversity**: Measures of diversity within individual samples (Observed OTUs, Shannon diversity)
- **Beta Diversity**: Measures of diversity between samples (Bray-Curtis, Unweighted/Weighted UniFrac)
- **Principal Coordinate Analysis (PCoA)**: Dimensionality reduction for beta diversity visualization
- **Statistical comparisons**: Age group comparisons and associations

## Data Sources
- Alpha diversity: `data/00_nhanes_omp_diversity_db/dada2rsv-alpha.txt`
- Beta diversity matrices: 
  - `data/00_nhanes_omp_diversity_db/dada2rsv-beta-braycurtis.txt`
  - `data/00_nhanes_omp_diversity_db/dada2rsv-beta-unwunifrac.txt`
  - `data/00_nhanes_omp_diversity_db/dada2rsv-beta-wunifrac.txt`
- Sample metadata: `data/00_nhanes_omp_abundance_db/nhanes_031725.sqlite`


In [1]:
# Load required libraries
library(phyloseq)
library(ggplot2)
library(dplyr)
library(tidyr)
library(readr)
library(vegan)
library(ape)
library(RColorBrewer)
library(viridis)
library(cowplot)
library(ggbeeswarm)
library(egg)

# Set working directory and paths
setwd("/Users/byeongyeoncho/main/github/nhanes_oral_mirco_cho")

# Define paths (following the pattern from reference files)
base_path <- "/Users/byeongyeoncho/main/github/nhanes_oral_mirco_cho"
phyloseq_obj_files_path <- file.path(base_path, "results/analyses_results/02_preprocess_db_n_phyloseq_out/intermediate")
results_path <- "results"
figures_path <- file.path(results_path, "diversity_analyses")

# Create results directory if it doesn't exist
if (!dir.exists(figures_path)) {
  dir.create(figures_path, recursive = TRUE)
}

# Set ggplot theme and font size
theme_set(egg::theme_article())
options(ggplot2.discrete.colour = function(...) scale_colour_viridis_d(...))
options(ggplot2.discrete.fill = function(...) scale_fill_viridis_d(...))

# Function to save plots
save_pdf_figure <- function(plot, filename, width = 7, height = 5) {
  ggsave(filename = file.path(figures_path, paste0(filename, ".pdf")), 
         plot = plot, width = width, height = height, dpi = 300)
  print(paste("Saved:", filename, ".pdf"))
}


ERROR: Error in library(phyloseq): there is no package called ‘phyloseq’


In [ ]:
# Load phyloseq object (following the pattern from reference files)
cat("Loading phyloseq object...\n")

# Load the ubiome relative abundance phyloseq object
ubiome_relative <- readRDS(file.path(phyloseq_obj_files_path, "ubiome_relative_updated.rds"))
cat("Phyloseq object loaded from:", file.path(phyloseq_obj_files_path, "ubiome_relative_updated.rds"), "\n")
cat("Samples:", nsamples(ubiome_relative), "Taxa:", ntaxa(ubiome_relative), "\n")

# Extract sample metadata from phyloseq object
sample_metadata <- data.frame(sample_data(ubiome_relative))
cat("Sample metadata dimensions:", dim(sample_metadata), "\n")
cat("Sample metadata columns:", head(colnames(sample_metadata), 10), "\n")

# Calculate alpha diversity metrics from phyloseq object
cat("Calculating alpha diversity metrics...\n")

# Observed OTUs (richness)
observed_otus <- estimate_richness(ubiome_relative, measures = "Observed")
observed_otus$SEQN <- rownames(observed_otus)

# Shannon diversity
shannon_div <- estimate_richness(ubiome_relative, measures = "Shannon")
shannon_div$SEQN <- rownames(shannon_div)

# Inverse Simpson
inv_simpson <- estimate_richness(ubiome_relative, measures = "InvSimpson")
inv_simpson$SEQN <- rownames(inv_simpson)

# Combine alpha diversity metrics
alpha_diversity <- data.frame(
  SEQN = observed_otus$SEQN,
  Observed_OTUs = observed_otus$Observed,
  Shannon_Diversity = shannon_div$Shannon,
  Inverse_Simpson = inv_simpson$InvSimpson
)

cat("Alpha diversity calculated. Dimensions:", dim(alpha_diversity), "\n")
print(head(alpha_diversity))


In [ ]:
# Calculate beta diversity matrices from phyloseq object
cat("Calculating beta diversity matrices...\n")

# Get OTU table from phyloseq object
otu_table <- otu_table(ubiome_relative)
if (taxa_are_rows(ubiome_relative)) {
  otu_table <- t(otu_table)  # Transpose if taxa are rows
}

# Calculate Bray-Curtis distance
cat("Calculating Bray-Curtis distances...\n")
bray_curtis_dist <- vegdist(otu_table, method = "bray")

# Calculate Unweighted UniFrac distance
cat("Calculating Unweighted UniFrac distances...\n")
unweighted_unifrac_dist <- UniFrac(ubiome_relative, weighted = FALSE)

# Calculate Weighted UniFrac distance  
cat("Calculating Weighted UniFrac distances...\n")
weighted_unifrac_dist <- UniFrac(ubiome_relative, weighted = TRUE)

cat("Beta diversity matrices calculated successfully\n")
cat("Bray-Curtis distance matrix size:", length(bray_curtis_dist), "\n")
cat("Unweighted UniFrac distance matrix size:", length(unweighted_unifrac_dist), "\n")
cat("Weighted UniFrac distance matrix size:", length(weighted_unifrac_dist), "\n")


In [ ]:
# Sample metadata is already loaded from phyloseq object above
# No need to load from SQLite database - it's already in the phyloseq object
cat("Sample metadata already extracted from phyloseq object\n")
cat("Sample metadata dimensions:", dim(sample_metadata), "\n")
cat("Sample metadata columns:", head(colnames(sample_metadata), 10), "\n")
print(head(sample_metadata, 5))


In [ ]:
# Merge alpha diversity data with sample metadata
cat("Merging alpha diversity with sample metadata...\n")

# Merge alpha diversity with metadata
merged_data <- merge(alpha_diversity, sample_metadata, by = "SEQN", all.x = TRUE)

# Create age groups (following the pattern from reference files)
# Check if age column exists and create age groups
age_cols <- colnames(merged_data)[grepl("age", colnames(merged_data), ignore.case = TRUE)]
cat("Age-related columns:", age_cols, "\n")

if (length(age_cols) > 0) {
  age_col <- age_cols[1]
  cat("Using age column:", age_col, "\n")
  
  # Create age groups (following the pattern from 5_age_analysis.Rmd)
  age_breaks <- c(14, seq(20, 85, by = 5))
  age_labels <- c("14-19", paste(seq(20, 75, by = 5), seq(24, 79, by = 5), sep = "-"), "80-85")
  merged_data$age_group <- cut(merged_data[[age_col]], 
                              breaks = age_breaks,
                              labels = age_labels,
                              right = FALSE)
} else {
  # If no age column found, create a dummy age group
  cat("No age column found, creating dummy age groups\n")
  merged_data$age_group <- "Unknown"
}

# Remove samples with missing diversity data
merged_data <- merged_data[complete.cases(merged_data[, c("Observed_OTUs", "Shannon_Diversity")]), ]

cat("Final merged data dimensions:", dim(merged_data), "\n")
cat("Age group distribution:\n")
print(table(merged_data$age_group, useNA = "ifany"))
print(head(merged_data[, c("SEQN", "Observed_OTUs", "Shannon_Diversity", "age_group")], 10))


In [ ]:
# Alpha Diversity Analysis
cat("Performing alpha diversity analysis...\n")

# 1. Observed OTUs vs Age Group
p1 <- ggplot(merged_data, aes(x = age_group, y = Observed_OTUs, fill = age_group)) +
  geom_boxplot(alpha = 0.7) +
  geom_beeswarm(alpha = 0.6, size = 0.8) +
  labs(title = "Observed OTUs by Age Group",
       x = "Age Group", 
       y = "Observed OTUs") +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 45, hjust = 1),
        text = element_text(size = 5))

# 2. Shannon Diversity vs Age Group  
p2 <- ggplot(merged_data, aes(x = age_group, y = Shannon_Diversity, fill = age_group)) +
  geom_boxplot(alpha = 0.7) +
  geom_beeswarm(alpha = 0.6, size = 0.8) +
  labs(title = "Shannon Diversity by Age Group",
       x = "Age Group", 
       y = "Shannon Diversity") +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 45, hjust = 1),
        text = element_text(size = 5))

# 3. Observed OTUs vs Shannon Diversity
p3 <- ggplot(merged_data, aes(x = Observed_OTUs, y = Shannon_Diversity, color = age_group)) +
  geom_point(alpha = 0.7, size = 1) +
  geom_smooth(method = "lm", se = TRUE, alpha = 0.3) +
  labs(title = "Observed OTUs vs Shannon Diversity",
       x = "Observed OTUs", 
       y = "Shannon Diversity",
       color = "Age Group") +
  theme(text = element_text(size = 5))

# Save individual plots
save_pdf_figure(p1, "alpha_diversity_observed_otus_by_age")
save_pdf_figure(p2, "alpha_diversity_shannon_by_age") 
save_pdf_figure(p3, "alpha_diversity_observed_vs_shannon")

# Combine plots
combined_alpha <- plot_grid(p1, p2, p3, ncol = 2, labels = c("A", "B", "C"))
save_pdf_figure(combined_alpha, "alpha_diversity_combined", width = 12, height = 8)

print("Alpha diversity plots saved successfully")


In [ ]:
# Beta Diversity Analysis - PCoA
cat("Performing beta diversity analysis...\n")

# Function to perform PCoA and create plots
perform_pcoa <- function(dist_matrix, title, color_var = NULL) {
  # Perform PCoA
  pcoa_result <- pcoa(dist_matrix)
  
  # Extract coordinates
  pcoa_coords <- pcoa_result$vectors[, 1:3]
  colnames(pcoa_coords) <- c("PCoA1", "PCoA2", "PCoA3")
  
  # Calculate explained variance
  explained_var <- pcoa_result$values$Eigenvalues / sum(pcoa_result$values$Eigenvalues) * 100
  
  # Create data frame for plotting
  pcoa_df <- data.frame(
    SEQN = rownames(pcoa_coords),
    PCoA1 = pcoa_coords[, 1],
    PCoA2 = pcoa_coords[, 2],
    PCoA3 = pcoa_coords[, 3]
  )
  
  # Merge with metadata if color_var is provided
  if (!is.null(color_var)) {
    pcoa_df <- merge(pcoa_df, merged_data[, c("SEQN", color_var)], by = "SEQN", all.x = TRUE)
  }
  
  # Create PCoA plot
  if (!is.null(color_var)) {
    p <- ggplot(pcoa_df, aes_string(x = "PCoA1", y = "PCoA2", color = color_var)) +
      geom_point(alpha = 0.7, size = 1) +
      labs(title = paste(title, "- PCoA Plot"),
           x = paste0("PCoA1 (", round(explained_var[1], 1), "%)"),
           y = paste0("PCoA2 (", round(explained_var[2], 1), "%)"),
           color = color_var) +
      theme(text = element_text(size = 5))
  } else {
    p <- ggplot(pcoa_df, aes(x = PCoA1, y = PCoA2)) +
      geom_point(alpha = 0.7, size = 1, color = "steelblue") +
      labs(title = paste(title, "- PCoA Plot"),
           x = paste0("PCoA1 (", round(explained_var[1], 1), "%)"),
           y = paste0("PCoA2 (", round(explained_var[2], 1), "%)")) +
      theme(text = element_text(size = 5))
  }
  
  return(list(plot = p, coords = pcoa_df, explained_var = explained_var))
}

# Perform PCoA for each distance matrix
cat("Computing PCoA for Bray-Curtis distances...\n")
bray_pcoa <- perform_pcoa(bray_curtis_dist, "Bray-Curtis", "age_group")

cat("Computing PCoA for Unweighted UniFrac distances...\n")
unweighted_pcoa <- perform_pcoa(unweighted_unifrac_dist, "Unweighted UniFrac", "age_group")

cat("Computing PCoA for Weighted UniFrac distances...\n")
weighted_pcoa <- perform_pcoa(weighted_unifrac_dist, "Weighted UniFrac", "age_group")

# Save individual PCoA plots
save_pdf_figure(bray_pcoa$plot, "beta_diversity_bray_curtis_pcoa")
save_pdf_figure(unweighted_pcoa$plot, "beta_diversity_unweighted_unifrac_pcoa")
save_pdf_figure(weighted_pcoa$plot, "beta_diversity_weighted_unifrac_pcoa")

# Combine PCoA plots
combined_pcoa <- plot_grid(bray_pcoa$plot, unweighted_pcoa$plot, weighted_pcoa$plot, 
                          ncol = 3, labels = c("A", "B", "C"))
save_pdf_figure(combined_pcoa, "beta_diversity_pcoa_combined", width = 15, height = 5)

print("Beta diversity PCoA plots saved successfully")


In [ ]:
# Statistical Analysis
cat("Performing statistical analysis...\n")

# Alpha diversity statistical tests
cat("Alpha diversity statistical tests:\n")

# Kruskal-Wallis test for Observed OTUs by age group
if (length(unique(merged_data$age_group)) > 1) {
  kw_observed <- kruskal.test(Observed_OTUs ~ age_group, data = merged_data)
  cat("Kruskal-Wallis test for Observed OTUs by age group:\n")
  cat("Chi-squared =", round(kw_observed$statistic, 3), 
      ", p-value =", round(kw_observed$p.value, 4), "\n")
  
  # Kruskal-Wallis test for Shannon diversity by age group
  kw_shannon <- kruskal.test(Shannon_Diversity ~ age_group, data = merged_data)
  cat("Kruskal-Wallis test for Shannon diversity by age group:\n")
  cat("Chi-squared =", round(kw_shannon$statistic, 3), 
      ", p-value =", round(kw_shannon$p.value, 4), "\n")
} else {
  cat("Only one age group found, skipping statistical tests\n")
}

# Beta diversity statistical tests
cat("\nBeta diversity statistical tests:\n")

# Function to perform PERMANOVA
perform_permanova <- function(dist_matrix, formula, data) {
  tryCatch({
    result <- adonis2(formula, data = data, permutations = 999)
    return(result)
  }, error = function(e) {
    cat("Error in PERMANOVA:", e$message, "\n")
    return(NULL)
  })
}

# Prepare data for PERMANOVA (samples that are in both distance matrix and metadata)
common_samples <- intersect(rownames(as.matrix(bray_curtis_dist)), merged_data$SEQN)
if (length(common_samples) > 0) {
  # Subset distance matrices to common samples
  bray_subset <- as.dist(as.matrix(bray_curtis_dist)[common_samples, common_samples])
  unweighted_subset <- as.dist(as.matrix(unweighted_unifrac_dist)[common_samples, common_samples])
  weighted_subset <- as.dist(as.matrix(weighted_unifrac_dist)[common_samples, common_samples])
  
  # Subset metadata to common samples
  meta_subset <- merged_data[merged_data$SEQN %in% common_samples, ]
  
  # Perform PERMANOVA
  if (length(unique(meta_subset$age_group)) > 1) {
    cat("PERMANOVA for Bray-Curtis distances:\n")
    permanova_bray <- perform_permanova(bray_subset, bray_subset ~ age_group, meta_subset)
    if (!is.null(permanova_bray)) print(permanova_bray)
    
    cat("PERMANOVA for Unweighted UniFrac distances:\n")
    permanova_unweighted <- perform_permanova(unweighted_subset, unweighted_subset ~ age_group, meta_subset)
    if (!is.null(permanova_unweighted)) print(permanova_unweighted)
    
    cat("PERMANOVA for Weighted UniFrac distances:\n")
    permanova_weighted <- perform_permanova(weighted_subset, weighted_subset ~ age_group, meta_subset)
    if (!is.null(permanova_weighted)) print(permanova_weighted)
  } else {
    cat("Only one age group found, skipping PERMANOVA tests\n")
  }
} else {
  cat("No common samples found between distance matrices and metadata\n")
}

print("Statistical analysis completed")


In [ ]:
# Additional Visualizations
cat("Creating additional visualizations...\n")

# 1. Distance to centroid analysis
if (length(common_samples) > 0 && length(unique(meta_subset$age_group)) > 1) {
  # Calculate centroids for each age group
  calculate_centroid_distances <- function(dist_matrix, groups) {
    # Convert distance matrix to data frame
    dist_df <- as.data.frame(as.matrix(dist_matrix))
    
    # Calculate mean distance to centroid for each group
    centroid_distances <- data.frame()
    
    for (group in unique(groups)) {
      group_samples <- names(groups)[groups == group]
      if (length(group_samples) > 1) {
        # Calculate distances within group
        group_dist <- dist_df[group_samples, group_samples]
        mean_dist <- mean(group_dist[upper.tri(group_dist)])
        
        centroid_distances <- rbind(centroid_distances, 
                                  data.frame(age_group = group, 
                                            mean_distance = mean_dist,
                                            n_samples = length(group_samples)))
      }
    }
    return(centroid_distances)
  }
  
  # Calculate centroid distances for each distance matrix
  bray_centroids <- calculate_centroid_distances(bray_subset, meta_subset$age_group)
  unweighted_centroids <- calculate_centroid_distances(unweighted_subset, meta_subset$age_group)
  weighted_centroids <- calculate_centroid_distances(weighted_subset, meta_subset$age_group)
  
  # Plot centroid distances
  p_centroids <- ggplot(bray_centroids, aes(x = age_group, y = mean_distance, fill = age_group)) +
    geom_bar(stat = "identity", alpha = 0.7) +
    labs(title = "Mean Distance to Centroid by Age Group (Bray-Curtis)",
         x = "Age Group", 
         y = "Mean Distance to Centroid") +
    theme(legend.position = "none",
          axis.text.x = element_text(angle = 45, hjust = 1),
          text = element_text(size = 5))
  
  save_pdf_figure(p_centroids, "beta_diversity_centroid_distances")
}

# 2. Alpha diversity summary statistics
alpha_summary <- merged_data %>%
  group_by(age_group) %>%
  summarise(
    n_samples = n(),
    mean_observed = mean(Observed_OTUs, na.rm = TRUE),
    sd_observed = sd(Observed_OTUs, na.rm = TRUE),
    mean_shannon = mean(Shannon_Diversity, na.rm = TRUE),
    sd_shannon = sd(Shannon_Diversity, na.rm = TRUE),
    .groups = 'drop'
  )

cat("Alpha diversity summary by age group:\n")
print(alpha_summary)

# 3. Create a summary table plot
library(gridExtra)
library(grid)

# Create summary table as a plot
summary_table <- alpha_summary %>%
  mutate(
    Observed_OTUs = paste0(round(mean_observed, 1), " ± ", round(sd_observed, 1)),
    Shannon_Diversity = paste0(round(mean_shannon, 2), " ± ", round(sd_shannon, 2))
  ) %>%
  select(age_group, n_samples, Observed_OTUs, Shannon_Diversity)

# Convert to table grob
table_grob <- tableGrob(summary_table, 
                       rows = NULL,
                       theme = ttheme_minimal(base_size = 5))

# Save summary table
pdf(file.path(figures_path, "alpha_diversity_summary_table.pdf"), 
    width = 8, height = 4)
grid.draw(table_grob)
dev.off()

print("Additional visualizations completed")
print(paste("All plots saved to:", figures_path))


## Summary

This notebook provides a comprehensive analysis of oral microbiome diversity data from the NHANES study, including:

### Alpha Diversity Analysis
- **Observed OTUs**: Number of unique operational taxonomic units per sample
- **Shannon Diversity**: Measure of species richness and evenness
- **Inverse Simpson**: Alternative diversity measure
- **Age Group Comparisons**: Statistical testing of diversity differences across age groups

### Beta Diversity Analysis  
- **Bray-Curtis Dissimilarity**: Abundance-based distance measure
- **Unweighted UniFrac**: Phylogenetic distance measure (presence/absence)
- **Weighted UniFrac**: Phylogenetic distance measure (abundance-weighted)
- **Principal Coordinate Analysis (PCoA)**: Dimensionality reduction for visualization

### Statistical Tests
- **Kruskal-Wallis**: Non-parametric test for alpha diversity differences
- **PERMANOVA**: Permutational multivariate analysis of variance for beta diversity
- **Centroid Analysis**: Distance to group centroids for beta diversity

### Output Files
All plots are saved as PDF files in the `results/diversity_analyses/` directory with standardized formatting using `egg::theme_article()` and font size 5.

### Key Findings
The analysis reveals patterns of microbial diversity across different age groups, providing insights into how the oral microbiome changes throughout the human lifespan.
